In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import src.config as cfg
from src.data_loader import load_bars, split_train_test
from src.indicators import add_indicators
from src.strategy import generate_signals
from src.backtest import run_backtest
from src.reporting import save_iteration, run_monte_carlo
import json

VERSION = "V2"
DATA_PATH = "data/NQ_1min.csv"


In [2]:
df = load_bars(DATA_PATH)
train, test = split_train_test(df, cfg.TRAIN_RATIO)
print(f"Total: {len(df):,}  Train: {len(train):,}  Test: {len(test):,}")


Total: 2,562,493  Train: 2,049,994  Test: 512,499


In [3]:
train = add_indicators(train, cfg)
print("Indicator columns:", [c for c in train.columns if c not in ['time','open','high','low','close','volume','session_break']])
train.head(3)


Indicator columns: ['ema_fast', 'ema_slow', 'ema_spread', 'ema_cross_up', 'ema_cross_down', 'zscore', 'atr', 'volume_zscore']


,time,open,high,low,close,volume,session_break,ema_fast,ema_slow,ema_spread,ema_cross_up,ema_cross_down,zscore,atr,volume_zscore
0,2019-01-01 17:01:00,6352.50,6364.75,6351.50,6361.25,296.0,False,6361.250000,6361.250000,0.000000,False,False,NaN,NaN,NaN
1,2019-01-01 17:02:00,6360.75,6371.50,6359.25,6368.50,339.0,False,6362.861111,6361.909091,0.952020,True,False,NaN,NaN,NaN
2,2019-01-01 17:03:00,6369.25,6371.75,6366.00,6366.25,195.0,False,6363.614198,6362.303719,1.310479,False,False,NaN,NaN,NaN


In [4]:
train = generate_signals(train, cfg)
n_long = (train.signal == 1).sum()
n_short = (train.signal == -1).sum()
print(f"Signals — long: {n_long:,}  short: {n_short:,}")


Signals — long: 50,222  short: 50,222

In [5]:
results = run_backtest(train, cfg, version=VERSION)
trades = results["trades"]
n = results["n_trades"]
wins = sum(1 for t in trades if t["label_win"])
print(f"Trades: {n:,}  Wins: {wins:,}  Win rate: {wins/n*100:.1f}%  Final equity: ${results['final_equity']:,.2f}")


Trades: 53,259  Wins: 544  Win rate: 1.0%  Final equity: $793.00


In [6]:
out_dir = save_iteration(VERSION, results, train, cfg)
print(f"Artifacts saved to: {out_dir}")


Saved iteration V2 to C:\Users\kunpa\Downloads\Projects\intra\iterations\V2
Artifacts saved to: C:\Users\kunpa\Downloads\Projects\intra\iterations\V2


In [7]:
mc_path = out_dir / "monte_carlo.json"
with open(mc_path) as f:
    mc = json.load(f)
print(f"Monte Carlo ({mc['n_sims']:,} sims, seed={mc['seed']})")
print(f"  Max Drawdown p50: ${mc['max_drawdown']['p50']:.2f}  p90: ${mc['max_drawdown']['p90']:.2f}  p99: ${mc['max_drawdown']['p99']:.2f}")
print(f"  VaR (5th pct): ${mc['var_trade_pnl']:.2f}  CVaR: ${mc['cvar_trade_pnl']:.2f}")
print(f"  Risk of ruin: {mc['risk_of_ruin_prob']*100:.1f}%")


Monte Carlo (10,000 sims, seed=42)
  Max Drawdown p50: $1816.00  p90: $2861.60  p99: $3714.52
  VaR (5th pct): $0.00  CVaR: $-0.23
  Risk of ruin: 91.0%


In [8]:
tlog = pd.read_csv(out_dir / "trader_log.csv")

def color_rows(row):
    if row["net_pnl_dollars"] > 0:
        return ["background-color: #d4edda"] * len(row)
    elif row["net_pnl_dollars"] < 0:
        return ["background-color: #f8d7da"] * len(row)
    else:
        return [""] * len(row)

styled = tlog.head(50).style.apply(color_rows, axis=1)
styled


,entry_time,side,entry_fill_price,sl_price,tp_price,net_pnl_dollars,cumulative_net_pnl_dollars
0,2019-01-01 17:03:00,long,6369.250000,6349.250000,6429.250000,21.000000,21.000000
1,2019-01-01 18:10:00,long,6376.500000,6356.500000,6436.500000,-10.000000,11.000000
2,2019-01-01 18:17:00,long,6376.500000,6356.500000,6436.500000,1.000000,12.000000
3,2019-01-01 18:55:00,long,6379.750000,6359.750000,6439.750000,-17.000000,-5.000000
4,2019-01-01 19:11:00,long,6380.250000,6360.250000,6440.250000,-11.000000,-16.000000
5,2019-01-01 20:58:00,long,6308.000000,6288.000000,6368.000000,-16.000000,-32.000000
6,2019-01-01 21:23:00,long,6303.750000,6283.750000,6363.750000,-10.000000,-42.000000
7,2019-01-01 21:35:00,long,6302.750000,6282.750000,6362.750000,-6.000000,-48.000000
8,2019-01-01 21:56:00,long,6306.250000,6286.250000,6366.250000,-20.000000,-68.000000
9,2019-01-01 22:00:00,long,6304.000000,6284.000000,6364.000000,-32.000000,-100.000000


In [9]:
html_path = out_dir / "trader_log.html"
tlog["entry_time"] = pd.to_datetime(tlog["entry_time"])
tlog_recent = tlog[tlog["entry_time"] >= "2024-03-01"]
tlog_recent.style.apply(color_rows, axis=1).to_html(html_path)
print(f"Styled HTML saved: {html_path}  ({len(tlog_recent)} trades from 2024-03-01)")


Styled HTML saved: C:\Users\kunpa\Downloads\Projects\intra\iterations\V2\trader_log.html  (6025 trades from 2024-03-01)
